# RAG Application

This notebook is designed for **third-year students**.

## Learning flow

```text
PDF Upload
    ↓
Load and extract text with PyMuPDF
    ↓
View page content and metadata
    ↓
Split text into smaller chunks
    ↓
Create Mistral embeddings
    ↓
Store chunks in Chroma vector database
    ↓
Retrieve relevant chunks
    ↓
Add retrieved context to a structured prompt
    ↓
Invoke Mistral LLM
    ↓
Generate the final answer
    ↓
Create a Gradio chat application
```

### Before running

- Use a **text-based PDF** containing information about Dr. M.G.R. University.
- Obtain a **Mistral AI API key**.
- Run the notebook cells from top to bottom.

## 1. Install the required packages

This cell installs only the libraries required by the notebook.

In [1]:
!pip -q install pymupdf langchain-community langchain-text-splitters langchain-mistralai langchain-chroma chromadb gradio pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.

## 2. Upload the university PDF

In [2]:
PDF_PATH = "/content/mgr.pdf"

## 3. Load and extract text using PyMuPDF

PyMuPDF opens the PDF and extracts the text page by page.

In [3]:
import fitz

pdf_document = fitz.open(PDF_PATH)

page_texts = []

for page_number, page in enumerate(pdf_document, start=1):
    page_text = page.get_text("text").strip()
    page_texts.append(page_text)

    print("=" * 80)
    print(f"PAGE {page_number}")
    print("=" * 80)
    print(page_text[:1000])

pdf_document.close()

full_text = "\n".join(page_texts)

print("\nTotal pages:", len(page_texts))
print("Total extracted characters:", len(full_text))

PAGE 1
Dr. M.G.R. University - Curated RAG Knowledge Base
1. Dr. M.G.R. Engineering College was established in Chennai in 1988.
2. Thai Moogambigai Dental College and Hospital was established in 1991.
3. Both institutions were conferred Deemed-to-be University status in 2003 under the name Dr.
M.G.R. Educational and Research Institute.
4. The deemed-to-be university status was granted with approval from the University Grants
Commission and the Government of India.
5. A.C.S. Medical College and Hospital became part of the university timeline in 2008.
6. The institution received graded autonomy status in 2018.
7. RajaRajeswari Medical College and Hospital, Bengaluru, appears in the university timeline
from 2019.
8. Sri Lalithambigai Medical College and Hospital, Chennai, appears in the university timeline
from 2021.
9. The Arni Off-Campus appears in the university timeline from 2024.
10. The University describes itself as a multidisciplinary hub covering engineering, medicine,
dentistry,

## 4. Convert the PDF into LangChain documents

`PyMuPDFLoader` creates one LangChain `Document` for each page.

Each document contains:

- `page_content`: text extracted from the page
- `metadata`: information such as source file, page number, title, author, and creation date

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(PDF_PATH)
page_documents = loader.load()

print("Number of LangChain page documents:", len(page_documents))
print("\nFirst page content:")
print(page_documents[0].page_content[:1000])

/tmp/ipykernel_907/3247467609.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Number of LangChain page documents: 2

First page content:
Dr. M.G.R. University - Curated RAG Knowledge Base
1. Dr. M.G.R. Engineering College was established in Chennai in 1988.
2. Thai Moogambigai Dental College and Hospital was established in 1991.
3. Both institutions were conferred Deemed-to-be University status in 2003 under the name Dr.
M.G.R. Educational and Research Institute.
4. The deemed-to-be university status was granted with approval from the University Grants
Commission and the Government of India.
5. A.C.S. Medical College and Hospital became part of the university timeline in 2008.
6. The institution received graded autonomy status in 2018.
7. RajaRajeswari Medical College and Hospital, Bengaluru, appears in the university timeline
from 2019.
8. Sri Lalithambigai Medical College and Hospital, Chennai, appears in the university timeline
from 2021.
9. The Arni Off-Campus appears in the university timeline from 2024.
10. The University describes itself as a multidiscipl

## 5. Display the PDF metadata

Metadata helps us identify where a retrieved chunk came from.

In [5]:
import pandas as pd

metadata_rows = []

for index, document in enumerate(page_documents):
    metadata_rows.append({
        "document_index": index,
        "page_number": document.metadata.get("page", index) + 1,
        "source": document.metadata.get("source"),
        "title": document.metadata.get("title"),
        "author": document.metadata.get("author"),
        "total_pages": document.metadata.get("total_pages"),
        "all_metadata": document.metadata
    })

metadata_table = pd.DataFrame(metadata_rows)
display(metadata_table)

,document_index,page_number,source,title,author,total_pages,all_metadata
0,0,1,/content/mgr.pdf,Dr. M.G.R. University - Curated RAG Knowledge ...,OpenAI,2,{'producer': 'ReportLab PDF Library - (opensou...
1,1,2,/content/mgr.pdf,Dr. M.G.R. University - Curated RAG Knowledge ...,OpenAI,2,{'producer': 'ReportLab PDF Library - (opensou...


# Part A — Data Splitting

A complete PDF page may be too large for retrieval.  
Therefore, the text is divided into smaller pieces called **chunks**.

We will study only two splitters:

1. `CharacterTextSplitter`
2. `RecursiveCharacterTextSplitter`

## 6. Splitting example 1 — CharacterTextSplitter

This splitter divides text using a selected separator.  
Here, a space is used as the separator.

In [6]:
from langchain_text_splitters import CharacterTextSplitter

character_splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    add_start_index=True
)

character_chunks = character_splitter.split_documents(page_documents)

print("Number of character chunks:", len(character_chunks))

for index, chunk in enumerate(character_chunks[:3]):
    print("\n" + "=" * 80)
    print("CHARACTER CHUNK:", index)
    print("Metadata:", chunk.metadata)
    print("Text:", chunk.page_content[:500])

Number of character chunks: 4

CHARACTER CHUNK: 0
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-16T13:55:40+00:00', 'source': '/content/mgr.pdf', 'file_path': '/content/mgr.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': 'Dr. M.G.R. University - Curated RAG Knowledge Base', 'author': 'OpenAI', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-07-16T13:55:40+00:00', 'trapped': '', 'modDate': "D:20260716135540+00'00'", 'creationDate': "D:20260716135540+00'00'", 'page': 0, 'start_index': 0}
Text: Dr. M.G.R. University - Curated RAG Knowledge Base
1. Dr. M.G.R. Engineering College was established in Chennai in 1988.
2. Thai Moogambigai Dental College and Hospital was established in 1991.
3. Both institutions were conferred Deemed-to-be University status in 2003 under the name Dr.
M.G.R. Educational and Research Institute.
4. The deemed-to-be university status was granted with approval from the University

## 7. Splitting example 2 — RecursiveCharacterTextSplitter

This splitter tries several separators in order:

```text
paragraph → new line → space → character
```

It usually preserves the natural structure of the text better, so we will use it for the final RAG application.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    add_start_index=True
)

recursive_chunks = recursive_splitter.split_documents(page_documents)

print("Number of recursive chunks:", len(recursive_chunks))

for index, chunk in enumerate(recursive_chunks[:3]):
    print("\n" + "=" * 80)
    print("RECURSIVE CHUNK:", index)
    print("Metadata:", chunk.metadata)
    print("Text:", chunk.page_content[:500])

Number of recursive chunks: 4

RECURSIVE CHUNK: 0
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-16T13:55:40+00:00', 'source': '/content/mgr.pdf', 'file_path': '/content/mgr.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': 'Dr. M.G.R. University - Curated RAG Knowledge Base', 'author': 'OpenAI', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-07-16T13:55:40+00:00', 'trapped': '', 'modDate': "D:20260716135540+00'00'", 'creationDate': "D:20260716135540+00'00'", 'page': 0, 'start_index': 0}
Text: Dr. M.G.R. University - Curated RAG Knowledge Base
1. Dr. M.G.R. Engineering College was established in Chennai in 1988.
2. Thai Moogambigai Dental College and Hospital was established in 1991.
3. Both institutions were conferred Deemed-to-be University status in 2003 under the name Dr.
M.G.R. Educational and Research Institute.
4. The deemed-to-be university status was granted with approval from the University

## 8. Prepare the final chunks

We select the recursive chunks and add a unique `chunk_id` to the metadata.

In [8]:
import pandas as pd

chunks = recursive_chunks

for index, chunk in enumerate(chunks):
    # Add an easy-to-read chunk ID.
    chunk.metadata["chunk_id"] = f"chunk_{index:03d}"

    # Chroma metadata should contain simple values only.
    clean_metadata = {}

    for key, value in chunk.metadata.items():
        if value is None:
            continue
        if isinstance(value, (str, int, float, bool)):
            clean_metadata[key] = value
        else:
            clean_metadata[key] = str(value)

    chunk.metadata = clean_metadata

chunk_rows = []

for chunk in chunks[:10]:
    chunk_rows.append({
        "chunk_id": chunk.metadata["chunk_id"],
        "page_number": chunk.metadata.get("page", 0) + 1,
        "start_index": chunk.metadata.get("start_index"),
        "characters": len(chunk.page_content),
        "text_preview": chunk.page_content[:180]
    })

display(pd.DataFrame(chunk_rows))

print("Total final chunks:", len(chunks))

,chunk_id,page_number,start_index,characters,text_preview
0,chunk_000,1,0,712,Dr. M.G.R. University - Curated RAG Knowledge ...
1,chunk_001,1,702,779,from 2019.\n8. Sri Lalithambigai Medical Colle...
2,chunk_002,1,1387,616,"13. The University promotes ethical values, cr..."
3,chunk_003,2,0,497,18. The 2026 admissions page presents the inst...


Total final chunks: 4


# Part B — Embeddings

An embedding converts text into a list of numbers.

Text with similar meaning should have embeddings that are close to each other in vector space.

## 9. Enter the Mistral API key

The key is entered securely and is not printed in the output.

In [9]:
import getpass
import os

if not os.getenv("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass(
        "Enter your Mistral API key: "
    )

print("Mistral API key is ready.")

Enter your Mistral API key: ··········
Mistral API key is ready.


## 10. Create an embedding for one sample chunk

This cell shows:

- the original chunk text;
- the embedding dimension;
- the first ten values of the embedding.

In [10]:
from langchain_mistralai import MistralAIEmbeddings

embedding_model = MistralAIEmbeddings(
    model="mistral-embed"
)

sample_text = chunks[0].page_content
sample_embedding = embedding_model.embed_query(sample_text)

print("Sample text:")
print(sample_text[:500])

print("\nEmbedding dimension:", len(sample_embedding))
print("First 10 embedding values:")
print(sample_embedding[:10])

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

Sample text:
Dr. M.G.R. University - Curated RAG Knowledge Base
1. Dr. M.G.R. Engineering College was established in Chennai in 1988.
2. Thai Moogambigai Dental College and Hospital was established in 1991.
3. Both institutions were conferred Deemed-to-be University status in 2003 under the name Dr.
M.G.R. Educational and Research Institute.
4. The deemed-to-be university status was granted with approval from the University Grants
Commission and the Government of India.
5. A.C.S. Medical College and Hospital

Embedding dimension: 1024
First 10 embedding values:
[-0.026153564453125, -0.0193634033203125, 0.037933349609375, -0.0158233642578125, 0.010894775390625, 0.029052734375, 0.03955078125, -0.0147705078125, 0.007503509521484375, 0.0303497314453125]


# Part C — Store the Embeddings in a Vector Database

We use **Chroma** as the vector database.

A simplified Chroma record contains:

```text
Record
 ├── ID
 ├── Document text
 ├── Metadata
 └── Embedding vector
```

## 11. Create the Chroma vector database

For every chunk, Chroma stores:

- chunk text;
- metadata;
- Mistral embedding;
- unique database ID.

The database is saved in the `mgr_chroma_db` folder.

In [11]:
import shutil
from langchain_chroma import Chroma

DB_PATH = "./mgr_chroma_db"
COLLECTION_NAME = "mgr_university_rag"

# Remove an older demo database to avoid duplicate records.
shutil.rmtree(DB_PATH, ignore_errors=True)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=DB_PATH,
    collection_metadata={"hnsw:space": "cosine"}
)

stored_ids = vector_db.get()["ids"]

print("Vector database created.")
print("Collection name:", COLLECTION_NAME)
print("Stored records:", len(stored_ids))
print("Database folder:", DB_PATH)

Vector database created.
Collection name: mgr_university_rag
Stored records: 4
Database folder: ./mgr_chroma_db


## 12. Inspect how data is stored in the database

Only the first few records are displayed because an embedding can contain many numbers.

In [12]:
import pandas as pd

database_data = vector_db.get(
    include=["documents", "metadatas", "embeddings"]
)

database_rows = []

records_to_show = min(5, len(database_data["ids"]))

for index in range(records_to_show):
    embedding_vector = database_data["embeddings"][index]

    database_rows.append({
        "database_id": database_data["ids"][index],
        "document_preview": database_data["documents"][index][:180],
        "metadata": database_data["metadatas"][index],
        "embedding_dimension": len(embedding_vector),
        "embedding_first_8_values": [
            round(float(value), 5)
            for value in embedding_vector[:8]
        ]
    })

database_structure_table = pd.DataFrame(database_rows)
display(database_structure_table)

,database_id,document_preview,metadata,embedding_dimension,embedding_first_8_values
0,ad9825bd-508a-43ea-92ad-4d5c1707703e,Dr. M.G.R. University - Curated RAG Knowledge ...,"{'creationdate': '2026-07-16T13:55:40+00:00', ...",1024,"[-0.02617, -0.01913, 0.03827, -0.01614, 0.0105..."
1,7c538b30-8ae1-4e70-9a83-b2167f015b4e,from 2019.\n8. Sri Lalithambigai Medical Colle...,"{'creationDate': 'D:20260716135540+00'00'', 'm...",1024,"[-0.01718, 0.00639, 0.03096, -0.02682, 0.02248..."
2,5662964c-dc7f-44e5-957c-1c0f6560dbb9,"13. The University promotes ethical values, cr...","{'chunk_id': 'chunk_002', 'file_path': '/conte...",1024,"[-0.03629, -0.00226, 0.02682, -0.00679, 0.0089..."
3,430e69be-1163-4b7a-9d82-6cba8433b50d,18. The 2026 admissions page presents the inst...,"{'moddate': '2026-07-16T13:55:40+00:00', 'crea...",1024,"[-0.04303, -0.01361, 0.02901, -0.03032, 0.0208..."


# Part D — Retrieval Mechanism

When a user asks a question:

1. The question is converted into an embedding.
2. Chroma compares the question embedding with stored chunk embeddings.
3. The closest chunks are selected.
4. These chunks become the context for the LLM.

Because this database uses cosine distance, a **smaller distance means a closer match**.

## 13. Retrieve the most relevant chunks for one query

In [13]:
import pandas as pd

query = "What facilities and learning resources are available at the university?"

# The query is also converted into an embedding.
query_embedding = embedding_model.embed_query(query)

print("Query:", query)
print("Query embedding dimension:", len(query_embedding))
print("First 8 query embedding values:", query_embedding[:8])

# Chroma compares the query embedding with stored chunk embeddings.
retrieval_results = vector_db.similarity_search_with_score(
    query=query,
    k=3
)

retrieval_rows = []

for rank, (document, distance) in enumerate(retrieval_results, start=1):
    retrieval_rows.append({
        "rank": rank,
        "cosine_distance": round(float(distance), 5),
        "chunk_id": document.metadata.get("chunk_id"),
        "page_number": document.metadata.get("page", 0) + 1,
        "text_preview": document.page_content[:300]
    })

display(pd.DataFrame(retrieval_rows))

Query: What facilities and learning resources are available at the university?
Query embedding dimension: 1024
First 8 query embedding values: [-0.00765228271484375, 0.01273345947265625, 0.0312042236328125, -0.005802154541015625, 0.022613525390625, 0.030609130859375, 0.07781982421875, -0.024688720703125]


,rank,cosine_distance,chunk_id,page_number,text_preview
0,1,0.23710,chunk_002,1,"13. The University promotes ethical values, cr..."
1,2,0.28089,chunk_001,1,from 2019.\n8. Sri Lalithambigai Medical Colle...
2,3,0.29910,chunk_003,2,18. The 2026 admissions page presents the inst...


## 14. Build the augmented context

**Augmentation** means adding the retrieved PDF information to the user question before sending it to the LLM.

In [14]:
retrieved_documents = [
    document
    for document, distance in retrieval_results
]

context_parts = []

for index, document in enumerate(retrieved_documents, start=1):
    page_number = document.metadata.get("page", 0) + 1
    chunk_id = document.metadata.get("chunk_id", "unknown")

    context_parts.append(
        f"[Context {index} | Page {page_number} | {chunk_id}]\n"
        f"{document.page_content}"
    )

retrieved_context = "\n\n".join(context_parts)

print(retrieved_context)

[Context 1 | Page 1 | chunk_002]
13. The University promotes ethical values, creative ideas, and entrepreneurial skills for the
benefit of society and the nation.
14. The quality policy aims to develop a centre of excellence for education and research while
promoting competence, dignity, discipline, and humaneness.
15. The main campus is located at Maduravoyal in Chennai, Tamil Nadu.
16. The University states that its campus is Wi-Fi enabled and that departments have
laboratories, libraries, and research facilities.
17. The central library is located in the VOC Block at the Maduravoyal campus and provides
digital access and research support.

[Context 2 | Page 1 | chunk_001]
from 2019.
8. Sri Lalithambigai Medical College and Hospital, Chennai, appears in the university timeline
from 2021.
9. The Arni Off-Campus appears in the university timeline from 2024.
10. The University describes itself as a multidisciplinary hub covering engineering, medicine,
dentistry, paramedical and allied s

# Part E — Structured Prompt and LLM Invocation

A good RAG prompt should clearly separate:

- the assistant's role;
- rules;
- retrieved context;
- user question;
- expected answer style.

## 15. Create a structured RAG prompt

In [15]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        '''
You are a helpful assistant for Dr. M.G.R. Educational and Research Institute.

Follow these rules:
1. Answer only from the supplied university context.
2. Do not invent facts.
3. If the answer is not present in the context, say:
   "The uploaded university document does not contain this information."
4. Give a clear answer suitable for a university student.
5. Keep the answer concise.
'''
    ),
    (
        "human",
        '''
RETRIEVED UNIVERSITY CONTEXT:
{context}

STUDENT QUESTION:
{question}

FINAL ANSWER:
'''
    )
])

prompt_value = rag_prompt.invoke({
    "context": retrieved_context,
    "question": query
})

for message in prompt_value.messages:
    print("=" * 80)
    print("MESSAGE TYPE:", message.type)
    print(message.content)

MESSAGE TYPE: system

You are a helpful assistant for Dr. M.G.R. Educational and Research Institute.

Follow these rules:
1. Answer only from the supplied university context.
2. Do not invent facts.
3. If the answer is not present in the context, say:
   "The uploaded university document does not contain this information."
4. Give a clear answer suitable for a university student.
5. Keep the answer concise.

MESSAGE TYPE: human

RETRIEVED UNIVERSITY CONTEXT:
[Context 1 | Page 1 | chunk_002]
13. The University promotes ethical values, creative ideas, and entrepreneurial skills for the
benefit of society and the nation.
14. The quality policy aims to develop a centre of excellence for education and research while
promoting competence, dignity, discipline, and humaneness.
15. The main campus is located at Maduravoyal in Chennai, Tamil Nadu.
16. The University states that its campus is Wi-Fi enabled and that departments have
laboratories, libraries, and research facilities.
17. The central

## 16. Invoke the Mistral LLM

The LLM receives the structured prompt containing both the question and retrieved context.

In [16]:
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0,
    max_retries=2
)

llm_response = llm.invoke(prompt_value)

print("Generated answer:")
print(llm_response.content)

Generated answer:
The university provides the following facilities and learning resources:

1. **Campus Facilities**:
   - Wi-Fi enabled campus.
   - Laboratories, libraries, and research facilities in departments.
   - Central library located in the VOC Block at the Maduravoyal campus with digital access and research support.

2. **Academic Programs**:
   - Multidisciplinary hub covering engineering, medicine, dentistry, paramedical and allied sciences, architecture, law, management studies, science, and humanities.

3. **Accreditation and Support**:
   - NAAC A+ accredited deemed-to-be university (CGPA of 3.48).
   - Placement and Training Cell to connect students with career opportunities and align academics with industry needs.

4. **Values and Skills**:
   - Promotion of ethical values, creative ideas, and entrepreneurial skills.


# Part F — Complete RAG Generation Function

The following function performs the complete pipeline for any question:

```text
Question → Retrieval → Context → Prompt → LLM → Answer
```

## 17. Create the complete RAG function

In [17]:
def answer_with_rag(question, number_of_chunks=3):
    # Step 1: Retrieve relevant chunks.
    results = vector_db.similarity_search_with_score(
        query=question,
        k=number_of_chunks
    )

    # Step 2: Build the augmented context.
    context_parts = []
    source_lines = []

    for index, (document, distance) in enumerate(results, start=1):
        page_number = document.metadata.get("page", 0) + 1
        chunk_id = document.metadata.get("chunk_id", "unknown")

        context_parts.append(
            f"[Context {index} | Page {page_number} | {chunk_id}]\n"
            f"{document.page_content}"
        )

        source_lines.append(
            f"- Page {page_number}, {chunk_id}, "
            f"distance={float(distance):.4f}"
        )

    context = "\n\n".join(context_parts)

    # Step 3: Insert the context and question into the prompt.
    final_prompt = rag_prompt.invoke({
        "context": context,
        "question": question
    })

    # Step 4: Invoke Mistral.
    response = llm.invoke(final_prompt)

    # Step 5: Return the generated answer with retrieval details.
    sources = "\n".join(source_lines)

    return (
        response.content
        + "\n\n**Retrieved PDF chunks**\n"
        + sources
    )

## 18. Test the complete RAG function with one question

In [18]:
test_question = "What does the university provide for students?"

answer = answer_with_rag(test_question)

print("Question:", test_question)
print("\nAnswer:")
print(answer)

Question: What does the university provide for students?

Answer:
The university provides the following for students:

1. **Ethical values, creative ideas, and entrepreneurial skills** for societal and national benefit.
2. **A Wi-Fi enabled campus** with laboratories, libraries, and research facilities in departments.
3. **A central library** in the VOC Block at the Maduravoyal campus with digital access and research support.
4. **Multidisciplinary education** covering engineering, medicine, dentistry, paramedical and allied sciences, architecture, law, management studies, science, and humanities.
5. **Technically qualified, practically competent, and industry-ready learning** to suit professional and academic careers.
6. **Placement and Training Cell** to connect students with career opportunities and align academics with industry needs.
7. **NAAC A+ accreditation** (CGPA of 3.48) as a deemed-to-be university.

**Retrieved PDF chunks**
- Page 1, chunk_002, distance=0.2344
- Page 1, ch

## 19. Optional: Ask multiple questions in a loop

Type `exit` to stop the loop.

In [ ]:
while True:
    user_question = input("\nAsk a question about the university: ").strip()

    if user_question.lower() == "exit":
        print("Chat ended.")
        break

    if not user_question:
        print("Please enter a question.")
        continue

    print("\n", answer_with_rag(user_question))

# Part G — Gradio Chat Application

Gradio creates a simple browser-based chat interface.

The interface uses the same `answer_with_rag()` function.

## 20. Launch the Gradio RAG chatbot

In [19]:
import gradio as gr

def gradio_chat(message, history):
    if not message or not message.strip():
        return "Please enter a university-related question."

    return answer_with_rag(message.strip())

demo = gr.ChatInterface(
    fn=gradio_chat,
    title="Dr. M.G.R. University RAG Assistant",
    description=(
        "Ask questions based on the uploaded university PDF. "
        "The application retrieves relevant PDF chunks before generating an answer."
    ),
    examples=[
        "When was the university established?",
        "What facilities are available for students?",
        "What programmes are offered by the university?",
        "What is the vision of the university?"
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aab5e47607506653a0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Summary

You have implemented the complete RAG flow:

1. Uploaded a university PDF.
2. Extracted text using PyMuPDF.
3. Viewed page metadata.
4. Compared two text splitters.
5. Generated Mistral embeddings.
6. Stored text, metadata, and vectors in Chroma.
7. Retrieved relevant chunks for a query.
8. Added the chunks to a structured prompt.
9. Invoked a Mistral language model.
10. Generated a grounded answer.
11. Built a Gradio chat application.

## Important observation

The LLM is not asked to remember the whole PDF.  
For every question, the application first retrieves only the most relevant chunks and supplies them as context.